In [3]:
%%capture
!pip install sacrebleu
!pip install transformers==4.40.2
!pip install sentencepiece
!pip install indic-nlp-library
!pip install torch>=1.10.0
!pip install datasets
!pip install tqdm
!pip install bitsandbytes  # For 4-bit quantization of LLM
!pip install evaluate

In [43]:
import sacrebleu
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import logging as hf_logging
from huggingface_hub import list_repo_refs, notebook_login
from indicnlp.tokenize import indic_tokenize
from datasets import load_dataset, get_dataset_split_names, concatenate_datasets, DatasetDict
from tqdm import tqdm
import os

import time
import logging
import gc
import copy
import functools

import evaluate
from sacrebleu.metrics import CHRF

In [5]:
hf_logging.set_verbosity_error()

In [12]:
notebook_login()

In [50]:
DATASET_REPO = "umairhassan02/coco-captions-subset-59k-translated-v3"

In [31]:
def load_refs(branches):
  all_datasets = []

  for branch in tqdm(branches):
      try:
          ds = load_dataset(DATASET_REPO, revision=branch)
          all_datasets.append(ds)
      except Exception as e:
          print(f"Failed to load {branch}: {e}")
  return all_datasets

In [40]:
def combine_dataset_refs(all_datasets):
  combined_splits = {}

  for ds in tqdm(all_datasets):
      for split_name in ds.keys():  # usually 'train', 'validation', etc.
          if split_name not in combined_splits:
              combined_splits[split_name] = ds[split_name]
          else:
              combined_splits[split_name] = concatenate_datasets([combined_splits[split_name], ds[split_name]])

  # Create a final DatasetDict
  final_dataset = DatasetDict(combined_splits)
  return final_dataset

In [51]:
refs = list_repo_refs(DATASET_REPO, repo_type="dataset")
branches = [b.name for b in refs.branches if b.name != 'main']

In [53]:
all_datasets = load_refs(branches)

  3%|▎         | 2/62 [00:01<00:38,  1.56it/s]

README.md:   0%|          | 0.00/881 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/28.5M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/26.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2503 [00:00<?, ? examples/s]

...

 98%|█████████▊| 61/62 [05:05<00:05,  5.59s/it]

README.md:   0%|          | 0.00/760 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/29.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2746 [00:00<?, ? examples/s]

100%|██████████| 62/62 [05:09<00:00,  4.99s/it]


In [55]:
final_dataset = combine_dataset_refs(all_datasets)

100%|██████████| 62/62 [00:01<00:00, 33.47it/s]


In [56]:
FINAL_DATASET_REPO = "umairhassan02/coco-captions-subset-59k-translated-final"

In [57]:
final_dataset.push_to_hub(FINAL_DATASET_REPO)

Uploading the dataset shards:   0%|          | 0/31 [00:00<?, ? shards/s]

Map:   0%|          | 0/9449 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/95 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/84 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :   2%|1         | 1.41MB / 90.8MB            

CommitInfo(commit_url='https://huggingface.co/datasets/umairhassan02/coco-captions-subset-59k-translated-final/commit/216e5026a2f4597e1344375d3014405aeb6dcf6c', commit_message='Upload dataset', commit_description='', oid='216e5026a2f4597e1344375d3014405aeb6dcf6c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/umairhassan02/coco-captions-subset-59k-translated-final', endpoint='https://huggingface.co', repo_type='dataset', repo_id='umairhassan02/coco-captions-subset-59k-translated-final'), pr_revision=None, pr_num=None)